# Part 3 · Waste Description Classification\n**Owner: Jeff**  ·  Part 3 of the EcoSort Waste Management System\n\nTF-IDF + Logistic Regression baseline (Option A). Add DistilBERT for the top tier.

### Environment setup — run first (works in VS Code **and** Google Colab)
Detects where you're running, gets the dataset onto **local disk** (fast — no Google Drive), and unzips it.
In Colab it also clones the repo and installs requirements. Set `DATASET_URL` to the `realwaste.zip` GitHub **Release** link once the repo exists.

In [ ]:
import os, sys, zipfile, pathlib, shutil, subprocess

IN_COLAB = "google.colab" in sys.modules
DATASET_URL = ""  # <-- paste the GitHub Release asset URL for realwaste.zip

if IN_COLAB:
    if not os.path.exists("Ecosort-Waste-Management-System"):
        subprocess.run(["git","clone","https://github.com/Ted-star7/Ecosort-Waste-Management-System.git"], check=False)
    if os.path.isdir("Ecosort-Waste-Management-System"):
        os.chdir("Ecosort-Waste-Management-System")
    subprocess.run(["pip","-q","install","-r","requirements.txt"], check=False)
    if DATASET_URL and not os.path.exists("data/realwaste.zip"):
        subprocess.run(["wget","-q","-O","data/realwaste.zip",DATASET_URL], check=False)

# Unzip RealWaste -> data/RealWaste/<category>/  (idempotent)
if not pathlib.Path("data/RealWaste").exists() and pathlib.Path("data/realwaste.zip").exists():
    with zipfile.ZipFile("data/realwaste.zip") as z:
        z.extractall("data/_tmp")
    for p in pathlib.Path("data/_tmp").rglob("RealWaste"):
        shutil.move(str(p), "data/RealWaste"); break
    shutil.rmtree("data/_tmp", ignore_errors=True)

DATA_DIR = "data"
IMAGE_DIR = os.path.join(DATA_DIR, "RealWaste")
print("Setup OK. Images present:", os.path.isdir(IMAGE_DIR))


In [ ]:
import os, re, json, random, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(42); random.seed(42)

# The 9 categories are read from the folder names (sorted) so every part
# uses ONE identical, consistent label order:
IMG_HEIGHT = IMG_WIDTH = 224
BATCH_SIZE = 32

Needs `desc` / `train_txt` / `test_txt` / `clean_text` from Part 1 (or run the master top-to-bottom).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
pipe = Pipeline([("tfidf", TfidfVectorizer(preprocessor=clean_text, ngram_range=(1,2),
                                           min_df=2, max_features=20000, stop_words="english")),
                 ("clf", LogisticRegression(max_iter=1000, C=5.0, class_weight="balanced"))])
pipe.fit(train_txt["description"], train_txt["category"])

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
pred = pipe.predict(test_txt["description"])
print("Test accuracy:", accuracy_score(test_txt["category"], pred))
print(classification_report(test_txt["category"], pred))
sns.heatmap(confusion_matrix(test_txt["category"],pred,labels=class_names),annot=True,fmt="d",
            xticklabels=class_names,yticklabels=class_names)
plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()

In [ ]:
def classify_waste_description(description):
    """Classify a text description into a waste category (grader interface)."""
    return str(pipe.predict([description])[0])

classify_waste_description("greasy pizza box with cheese stuck on it")

### TODO (Jeff)\n- Inspect misclassifications / confusion cases\n- OPTIONAL Option B: fine-tune DistilBERT (all layers) and compare\n- Justify the architecture choice